<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/lab-cont-2026-2/blob/main/modulo2_plano_s/labs/lab05_plano_s_e_aproximacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Lab 05 — Regiões de Desempenho no Plano-s e Aproximações

**Módulo 02 · Semana 5 · Laboratório de Controle Automático (Ifes Guarapari)**

Neste laboratório você vai:
1. **Recriar** os gráficos $t_r \cdot \omega_n \times \zeta$ que geraram as aproximações do tempo de subida;
2. Desenhar as **regiões de desempenho** no plano-s e testar polos candidatos;
3. Validar a **regra dos 5×** para aproximação de 3ª por 2ª ordem;
4. Medir o **efeito do zero** (SPE e SPD) na resposta ao degrau.

Teoria de referência: `teoria_modulo2.md`, seção **§2.1**.

In [ ]:
# %% IMPORTS (rode esta célula primeiro)
!pip install --quiet control==0.10.2
import control as ct
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True

def step(G, t):
    t, y = ct.step_response(G, t)
    return np.asarray(t), np.asarray(y)   # python-control 0.10.2: converte NamedSignal

def medir(T, t=None):
    """Mede Mp (%), tp, tr(0-100%) e ts(5%) da resposta ao degrau de T."""
    if t is None: t = np.linspace(0, 50, 20000)
    t, y = step(T, t)
    yss = y[-1]
    Mp = 100*(y.max() - yss)/yss
    tp = t[np.argmax(y)]
    tr = t[np.argmax(y >= yss)]
    fora = np.where(np.abs(y - yss) > 0.05*abs(yss))[0]
    ts = t[fora[-1]] if len(fora) else 0.0
    return dict(Mp=Mp, tp=tp, tr=tr, ts=ts, yss=yss)

print('python-control', ct.__version__)

## Parte 1 — Recriando as aproximações do tempo de subida

Na teoria (§2.1.5) usamos $t_r(10\text{–}90\%) \approx 1{,}6/\omega_n$ e
$t_r(0\text{–}100\%) \approx 2{,}4/\omega_n$ — constantes que **valem para $\zeta \approx 0{,}5$**.
De onde vêm esses números? De medir $t_r$ em sistemas de 2ª ordem com vários $\zeta$ e
multiplicar por $\omega_n$ (normalização). **É esse gráfico que vamos recriar agora.**

Para cada $\zeta$, montamos $T(s) = \dfrac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$ com $\omega_n = 1$
(assim, o $t_r$ medido já é $t_r \cdot \omega_n$).

In [ ]:
zetas = np.arange(0.1, 1.0, 0.05)
tr1090, tr0100 = [], []
t = np.linspace(0, 30, 30000)
for z in zetas:
    T = ct.tf([1], [1, 2*z, 1])          # wn = 1
    tt, y = step(T, t)
    i10 = np.argmax(y >= 0.1); i90 = np.argmax(y >= 0.9)
    tr1090.append(tt[i90] - tt[i10])     # tr(10-90%)
    tr0100.append(tt[np.argmax(y >= 1.0)])  # tr(0-100%)

plt.plot(zetas, tr1090, 'o', ms=4, label='medido: tr(10–90%)·ωn')
plt.plot(zetas, tr0100, 's', ms=4, label='medido: tr(0–100%)·ωn')
plt.plot(zetas, 1.8*zetas**3 - 0.4*zetas**2 + zetas + 1, '-', label='1,8ξ³ − 0,4ξ² + ξ + 1')
plt.plot(zetas, 2.3*zetas**2 - 0.08*zetas + 1.1, '-', label='2,3ξ² − 0,08ξ + 1,1')
plt.axhline(1.6, color='gray', ls='--'); plt.axhline(2.4, color='gray', ls='--')
plt.text(0.72, 1.55, '1,6  (ξ ≈ 0,5)'); plt.text(0.72, 2.45, '2,4  (ξ ≈ 0,5)')
plt.xlabel('ζ'); plt.ylabel('tr·ωn'); plt.legend(fontsize=8); plt.title('Recriando as aproximações do tempo de subida');

**✏️ Responda no relatório:**
1. Para que faixa de $\zeta$ as aproximações polinomiais têm erro menor que 5 %? (Meça!)
2. Por que faz sentido usar as constantes 1,6 e 2,4 só quando $\zeta \approx 0{,}5$?
3. Se $\zeta = 0{,}3$ e você usou $1{,}6/\omega_n$ no projeto, o $t_r$ real será maior ou menor que o projetado?

## Parte 2 — Regiões de desempenho no plano-s

Cada requisito vira uma região (§2.1.3–§2.1.6): **overshoot → setor**, **$t_p$ → faixa**,
**$t_s$ → semirreta**, **$t_r$ → círculo**. Vamos desenhar a região combinada de
$M_p \leq 20\%$, $t_p \leq 3{,}14$ s e $t_s(2\%) \leq 8$ s e testar dois polos candidatos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
# setor: beta <= arccos(0.456) = 62.9°
beta = np.arccos(0.456); tan = np.tan(beta)
xs = np.linspace(-4, 0, 200)
ax.fill_between(xs, tan*xs, -tan*xs, alpha=0.25, color='C0', label='Mp ≤ 20% (β ≤ 62,9°)')
# faixa wd >= 1
ax.axhspan(1, 4, xmin=0, xmax=1, alpha=0.15, color='C1')
ax.axhspan(-4, -1, xmin=0, xmax=1, alpha=0.15, color='C1', label='tp ≤ 3,14 s (ωd ≥ 1)')
# semirreta sigma >= 0.5
ax.axvspan(-4, -0.5, alpha=0.12, color='C2', label='ts(2%) ≤ 8 s (σ ≥ 0,5)')
# candidatos
ax.plot(-0.5, 1, 'rx', ms=12, mew=3, label='□1 = −0,5+1j')
ax.plot(-0.6, 1.1, 'go', ms=12, mew=3, mfc='none', label='□2 = −0,6+1,1j')
ax.set_xlim(-4, 0.5); ax.set_ylim(-4, 4)
ax.set_xlabel('σ'); ax.set_ylabel('jω'); ax.legend(fontsize=8, loc='lower left')
ax.set_title('Região combinada: os polos dominantes devem estar na INTERSEÇÃO');

**✏️ Responda:** o ponto $-0{,}5+1j$ está na fronteira de duas regiões e mesmo assim falha. Qual requisito ele viola?
Confira calculando $\beta$ (compare com o Exercício 2.1.1).

## Parte 3 — A regra dos 5× (aproximação de 3ª por 2ª ordem)

$T(s) = \dfrac{4\,p_3}{(s^2+2s+4)(s+p_3)}$ — par dominante $-1 \pm 1{,}73j$ ($\zeta = 0{,}5$, $\omega_n = 2$),
mais um polo em $-p_3$ (ganho DC sempre 1). A 2ª ordem prevê $M_p = 16{,}3\%$ e $t_p = 1{,}81$ s.

In [ ]:
t = np.linspace(0, 12, 12000)
for p3 in [10, 5, 3, 1, 0.5]:
    T = ct.tf([4*p3], [1, 2+p3, 4+2*p3, 4*p3])
    m = medir(T, t)
    plt.plot(*step(T, t), label=f'p3 = {p3} ({p3}×σ): Mp = {m["Mp"]:.1f}%')
T2 = ct.tf([4], [1, 2, 4])
plt.plot(*step(T2, t), 'k--', label='2ª ordem pura (16,3%)')
plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.legend(fontsize=8)
plt.title('Regra dos 5×: p3 ≥ 5·|Re(par)| ⇒ aproximação confiável');

**✏️ Responda:**
1. Para $p_3 = 1$, por que o overshoot "morreu"? (Quem domina a resposta?)
2. Um 3º polo em $-10$ com par em $-3 \pm 3j$ satisfaz a regra dos 5×?

## Parte 4 — O efeito do zero

$T_1(s) = \dfrac{s+a}{a}\,T_0(s)$ com $T_0 = \dfrac{4}{s^2+2s+4}$ (ganho DC preservado).
No tempo: $y_1 = y + \dot y / a$ — o zero **soma a derivada** (§2.1.9).

In [ ]:
t = np.linspace(0, 10, 10000)
T0 = ct.tf([4], [1, 2, 4])
plt.plot(*step(T0, t), 'k-', lw=2, label='sem zero (16,3%)')
for a in [10, 5, 2]:
    T1 = ct.tf([4/a, 4], [1, 2, 4])
    m = medir(T1, t)
    plt.plot(*step(T1, t), label=f'zero −{a}: Mp = {m["Mp"]:.1f}%')
T1r = ct.tf([-4/5, 4], [1, 2, 4])   # zero em +5 (fase não mínima)
tt, yr = step(T1r, t)
plt.plot(tt, yr, label=f'zero +5 (SPD): undershoot {100*yr.min():.1f}%')
plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.legend(fontsize=8)
plt.title('Zero perto acelera e aumenta o overshoot; zero no SPD causa undershoot');

## Sua vez (entregue no relatório)

1. **Requisito inverso:** projete (no papel ou via simulação) uma região do plano-s para
   $M_p \leq 10\%$, $t_p \leq 0{,}5$ s e $t_s(5\%) \leq 2$ s. Um polo em $-2+6j$ atende?
2. **Quase-cancelamento:** simule $T(s) = \dfrac{(s+1{,}1)/1{,}1}{(s+1)(s^2+2s+4)} \cdot 4$ e compare
   com a 2ª ordem pura. O que acontece quando o zero quase cancela o polo $-1$? E com zero em $-0{,}9$?
3. **Zero RHP:** para o zero em $+2$, meça o undershoot e relacione com $|1/a|$.

**Critério de correção:** figuras legíveis + respostas curtas com números medidos (não "aumenta/diminui" sem valor).